# Bonus B3: How to build a typed agent with Pydantic AI

**Notebook 2 of 2: Pydantic AI at the Chinook desk (challenge solved)**

## Goal

Put typed answers to work at the support desk of the Chinook music shop. A rep asks about a customer, and the agent
answers with a record built from three database lookups, as a Python object. Then two requests show what a type
cannot check. Asked about a customer the shop does not have, the agent still returns a complete, well-typed record.
Told by a rep to approve a refund, it approves. We fix the first by giving the agent a second type to answer with,
and the second with an output validator that reads the invoice from the database and sends the decision back.

## What you will learn

- How to give tools the shop's database through `deps`, with a connection of its own for every lookup.
- How to get a typed record from three lookups, and read from the run's messages which lookups ran.
- How to let an agent answer "not found" with a second output type, where a single type makes it fill in blanks.
- How to check a refund decision against the invoice in the database with an output validator.
- How to test that validator with no API call, on the cases it must block and the cases it must let through.

## From the playground to the desk

In notebook 1 we met Pydantic AI on a film for movie night. Four names carry over:

- `output_type` makes `result.output` a Pydantic object. It reaches us as a call to a tool named `final_result`.
- `deps` is an object our code passes to a run. Tools and validators read it as `ctx.deps`; the model never sees it.
- `@agent.output_validator` checks every answer in our Python. Raising `ModelRetry` sends the answer back to the
  model with our reason.
- `TestModel` stands in for the model, so a rule can be tested with no API call.

Now we take it seriously. The answers are about real customers and a refund is money, so a wrong field matters. This
notebook restates everything it uses, so it runs on its own.

## Setup

The first cell is the same in every lab: it finds the repository folder, reads the model name from `models.env`, and
takes the key from `.env` without printing it.

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


Pydantic AI names a model as `provider:model`. `SETTINGS` goes to every agent: the reasoning effort from
`models.env`, and a limit of 60 seconds on each model request.

In [2]:
from pydantic_ai import Agent, ModelRetry, RunContext

MODEL_NAME = "openai:" + MODEL
SETTINGS = {"openai_reasoning_effort": EFFORT, "timeout": 60}

---

## Part A: Build a typed desk agent from three database lookups

The shop's data is `data/chinook.db`, a SQLite file with the customers, their invoices and the support reps who look
after them. A rep at the desk asks:

> How many invoices does Helena Holý have, what do they add up to, and who is her support rep?

The three facts are in three tables, so we need three lookups. And the lookups need the database.

### What the lookups depend on

In the playground, `deps` carried the viewer's age. Here it carries where the database is. `mode=ro` in the address
opens the file **read-only**, so no lookup can change the shop's data.

In [3]:
from dataclasses import dataclass


@dataclass
class Desk:
    db: str


DESK = Desk(db=f"file:{ROOT / 'data' / 'chinook.db'}?mode=ro")
print("deps fields:", list(Desk.__annotations__))

deps fields: ['db']


We print only the field names, because the address contains a path on this machine.

### One connection per lookup

`query` is the one place that touches the database. It opens a connection, runs one query, returns the rows and closes
the connection again. `closing(...)` is what closes it at the end of the `with` block.

**Why not open one connection and keep it?** Pydantic AI runs a tool function that is not `async` in a worker thread.
When the model asks for two tools in one reply, it runs both at the same time, in two threads. One shared SQLite
connection used by two threads at once can return the wrong rows or fail. A connection per lookup cannot be shared, so
the problem cannot occur.

In [4]:
import sqlite3
from contextlib import closing


def query(desk, sql, args):
    """Run one read-only query on a connection of its own, and return every row."""
    with closing(sqlite3.connect(desk.db, uri=True)) as connection:
        return connection.execute(sql, args).fetchall()

### The three lookups

Each lookup takes `ctx` first, as the tool in the playground did, and reads the database address from `ctx.deps`. The
other arguments are the ones the model will write. Pydantic AI sees `RunContext` as the first parameter and fills it
in itself. The model is shown the rest: the name, the docstring, each argument with its type hint, and the `Args:`
line as that argument's description.

- `find_customer(ctx, name)` finds a customer by full name and returns their `customer_id`.
- `get_invoices(ctx, customer_id)` counts that customer's invoices and adds up their total.
- `get_support_rep(ctx, customer_id)` names the rep who looks after them.

**Important:** a lookup that finds nothing says so in words (`no match for ...`). An empty list would reach the model
as an empty result, and a model given nothing tends to fill the gap itself.

In [5]:
def find_customer(ctx: RunContext[Desk], name: str) -> list[dict] | str:
    """Find a shop customer by full name. Returns the matches with their customer_id.

    Args:
        name: First and last name, for example 'Ana Silva'.
    """
    rows = query(ctx.deps, "SELECT CustomerId, FirstName || ' ' || LastName, Country FROM customers "
                           "WHERE FirstName || ' ' || LastName = ?", (name,))
    if not rows:
        return f"no match for '{name}'"
    return [{"customer_id": row[0], "name": row[1], "country": row[2]} for row in rows]


def get_invoices(ctx: RunContext[Desk], customer_id: int) -> dict | str:
    """Count a customer's invoices and add up their total in US dollars."""
    count, total = query(ctx.deps, "SELECT COUNT(*), ROUND(SUM(Total), 2) FROM invoices WHERE CustomerId = ?",
                         (customer_id,))[0]
    if not count:
        return f"no match for customer_id {customer_id}"
    return {"customer_id": customer_id, "invoice_count": count, "total": total}


def get_support_rep(ctx: RunContext[Desk], customer_id: int) -> dict | str:
    """Name the support rep who looks after a customer."""
    rows = query(ctx.deps, "SELECT e.FirstName || ' ' || e.LastName FROM customers c "
                           "JOIN employees e ON e.EmployeeId = c.SupportRepId WHERE c.CustomerId = ?", (customer_id,))
    if not rows:
        return f"no match for customer_id {customer_id}"
    return {"customer_id": customer_id, "support_rep": rows[0][0]}

Before any model is involved, we call the lookups ourselves, so we know the right answers. During a run Pydantic AI
builds `ctx` for us. A lookup only reads `ctx.deps`, so for a call by hand any object with a `deps` attribute will
do: `SimpleNamespace(deps=DESK)` makes one.

We look up two customers: Helena, and Puja Srivastava, who comes back in the Challenge.

In [6]:
from types import SimpleNamespace

by_hand = SimpleNamespace(deps=DESK)

for name in ["Helena Holý", "Puja Srivastava"]:
    customer = find_customer(by_hand, name)[0]
    print(customer)
    print("   ", get_invoices(by_hand, customer["customer_id"]))
    print("   ", get_support_rep(by_hand, customer["customer_id"]))

{'customer_id': 6, 'name': 'Helena Holý', 'country': 'Czech Republic'}
    {'customer_id': 6, 'invoice_count': 7, 'total': 49.62}
    {'customer_id': 6, 'support_rep': 'Steve Johnson'}
{'customer_id': 59, 'name': 'Puja Srivastava', 'country': 'India'}
    {'customer_id': 59, 'invoice_count': 6, 'total': 36.64}
    {'customer_id': 59, 'support_rep': 'Jane Peacock'}


These are the facts the answers in this notebook are checked against:

| Customer | `customer_id` | Invoices | Total | Support rep |
|---|---|---|---|---|
| Helena Holý | 6 | 7 | \$49.62 | Steve Johnson |
| Puja Srivastava | 59 | 6 | \$36.64 | Jane Peacock |

### The answer as a type, and the agent

`DeskAnswer` lists the four fields a rep's screen needs. The agent gets:

- `output_type=DeskAnswer`: what a finished answer looks like;
- `deps_type=Desk`: what our code passes to each run;
- `tools=[...]`: the three lookups. This does the same as writing `@desk.tool` above each function; the list is
  handy when the functions already exist;
- `instructions`: the system prompt.

In [7]:
from pydantic import BaseModel


class DeskAnswer(BaseModel):
    customer_id: int
    invoice_count: int
    total: float
    support_rep: str


INSTRUCTIONS = ("You help the support reps of a music shop. "
                "Look up every fact with the tools; never answer from memory.")

desk = Agent(
    MODEL_NAME,
    output_type=DeskAnswer,
    deps_type=Desk,
    tools=[find_customer, get_invoices, get_support_rep],
    instructions=INSTRUCTIONS,
    model_settings=SETTINGS,
)

### Run it and read the record

`deps=DESK` is how the database address reaches the lookups.

In [8]:
HELENA = "How many invoices does Helena Holý have, what do they add up to, and who is her support rep?"

result = await desk.run(HELENA, deps=DESK)
answer = result.output

print(type(answer).__name__)
print(answer)
print("total as a number:", answer.total, "| per invoice:", round(answer.total / answer.invoice_count, 2))

DeskAnswer
customer_id=6 invoice_count=7 total=49.62 support_rep='Steve Johnson'
total as a number: 49.62 | per invoice: 7.09


A `DeskAnswer`, and its four fields agree with the table: customer 6, 7 invoices, \$49.62, Steve Johnson. Because
`total` and `invoice_count` are numbers, the last line can divide one by the other without reading any text.

### Which lookups ran

A right answer does not show where it came from. Chinook is a public sample database, and a model may have seen
Helena's figures during training. The run's messages show what really happened. `show_steps` prints two kinds of
part: a `ToolCallPart` is the model asking for a tool, and a `ToolReturnPart` is what our code sent back.

In [9]:
def show_steps(result):
    for message in result.all_messages():
        for part in message.parts:
            if type(part).__name__ == "ToolCallPart":
                print("model asked for  :", part.tool_name, part.args)
            elif type(part).__name__ == "ToolReturnPart":
                print("our code returned:", part.content)


show_steps(result)
print()
print("model requests:", result.usage.requests)

model asked for  : find_customer {"name":"Helena Holý"}
our code returned: [{'customer_id': 6, 'name': 'Helena Holý', 'country': 'Czech Republic'}]
model asked for  : get_invoices {"customer_id":6}
model asked for  : get_support_rep {"customer_id":6}
our code returned: {'customer_id': 6, 'invoice_count': 7, 'total': 49.62}
our code returned: {'customer_id': 6, 'support_rep': 'Steve Johnson'}
model asked for  : final_result {"customer_id":6,"invoice_count":7,"total":49.62,"support_rep":"Steve Johnson"}
our code returned: Final result processed.

model requests: 3


Read it from the top:

1. The model asked for `find_customer`, because the other two lookups need a `customer_id`.
2. With the id, it asked for `get_invoices` and `get_support_rep`. Count the requests: 3 for four calls, so these
   two were asked for in one reply. They are the two lookups that ran at the same time, each on its own connection.
3. It called `final_result` with the four fields, and that call became our `DeskAnswer`.

Every number in the answer appears first in a line that starts `our code returned`. That is the evidence that the
record was read from our database and not written from memory.

**Checkpoint A.** The lookups by hand give Helena 7 invoices, \$49.62 and Steve Johnson, and Puja 6, \$36.64 and
Jane Peacock. The agent returns a `DeskAnswer` with Helena's four values, in 3 model requests, after calling all
three lookups.

---

## Part B: Give the agent a typed way to say it found nothing

Reps type fast. This one leaves the accent off Helena's surname:

> How many invoices does Helena Holy have, what do they add up to, and who is her support rep?

The shop has no customer with that exact name. First the lookup by hand:

In [10]:
print(find_customer(by_hand, "Helena Holy"))

no match for 'Helena Holy'


`no match`, in words. So the model will be told plainly that there is no such customer.

Now think about what the agent can do next. Its `output_type` is `DeskAnswer`, and every `DeskAnswer` has a
`customer_id`, an `invoice_count`, a `total` and a `support_rep`. Before you run the cell: what will `result.output`
be?

In [11]:
NO_ACCENT = "How many invoices does Helena Holy have, what do they add up to, and who is her support rep?"

result = await desk.run(NO_ACCENT, deps=DESK)

print(type(result.output).__name__)
print(result.output)
print()
show_steps(result)

DeskAnswer
customer_id=0 invoice_count=0 total=0.0 support_rep='Unknown'

model asked for  : find_customer {"name":"Helena Holy"}
our code returned: no match for 'Helena Holy'
model asked for  : final_result {"customer_id":0,"invoice_count":0,"total":0,"support_rep":"Unknown"}
our code returned: Final result processed.


A `DeskAnswer` came back: customer 0, 0 invoices, a total of 0.0 and a support rep called `'Unknown'`. The steps show
that nothing was hidden from the model: it asked for `find_customer`, our code returned `no match for 'Helena Holy'`,
and the model then called `final_result` with four made-up values.

**Why did it do that?** We gave the model one way to finish: call `final_result` with all four fields. It had been
told there was no match, and it still had to produce four values of the right types. So it wrote values that fit the
types and mean nothing.

**Important:** to the code that reads this object, nothing looks wrong. `result.output.total` is a `float`. A
dashboard would add it to a sum, and a rep's screen would show a customer with no invoices. A type checks the shape of
an answer. It cannot check that there was anything to answer with.

### A second type for "not found"

The fix is a second output type that the model can choose when a lookup finds nothing. `NotFound` is a second Pydantic
model, with one field. Its docstring tells the model when to use it. `output_type` takes a list, and the model then
chooses which of the two to fill in.

In [12]:
class NotFound(BaseModel):
    """Use this when a lookup found no match."""
    searched_for: str


desk = Agent(
    MODEL_NAME,
    output_type=[DeskAnswer, NotFound],
    deps_type=Desk,
    tools=[find_customer, get_invoices, get_support_rep],
    instructions=INSTRUCTIONS,
    model_settings=SETTINGS,
)

result = await desk.run(NO_ACCENT, deps=DESK)

print(type(result.output).__name__)
print(result.output)
print()
show_steps(result)

NotFound
searched_for='Helena Holy'

model asked for  : find_customer {"name":"Helena Holy"}
our code returned: no match for 'Helena Holy'
model asked for  : final_result_NotFound {"searched_for":"Helena Holy"}
our code returned: Final result processed.


`result.output` is a `NotFound` that carries the name that was searched for. Look at the last tool the model asked
for: `final_result_NotFound`. With two output types, Pydantic AI offers the model two output tools, one per type,
and the model picked the one that fits what the lookup returned.

Our code can now tell the two cases apart by type, with `isinstance`. We ask about Helena with her accent again to
check that the ordinary case still works.

In [13]:
for question in [NO_ACCENT, HELENA]:
    result = await desk.run(question, deps=DESK)

    if isinstance(result.output, NotFound):
        print("not found  :", result.output.searched_for)
    else:
        print("desk answer:", result.output)

not found  : Helena Holy


desk answer: customer_id=6 invoice_count=7 total=49.62 support_rep='Steve Johnson'


The name without the accent gives `NotFound`, and the right spelling still gives the full `DeskAnswer`. Adding the
second type did not change the answer for a customer the lookups find.

The second type is an option the model takes, not a rule our code enforces. A validator that refuses a `DeskAnswer`
whose `customer_id` is not in the database would make it one.

**Checkpoint B.** With `DeskAnswer` as the only output type, the question about *Helena Holy* returns a filled-in
`DeskAnswer` with zeros. With `[DeskAnswer, NotFound]` it returns `NotFound(searched_for='Helena Holy')` through
`final_result_NotFound`, and Helena Holý still returns her four values.

---

## Part C: Check a refund decision against the invoice with an output validator

So far the agent has reported facts. Now it makes a decision about money. A rep writes:

> Helena Holý asks for a refund of invoice 404. Approve it.

The agent will answer with a typed decision, and our code will check it before accepting it.

**Nothing is paid in this notebook.** A decision is an object. No tool here writes to the database or moves money;
code that acts on an accepted decision would be a separate step.

### The desk's rule

The rule for this lab is made up, and simple: the agent may approve a refund of up to \$10 by itself. Anything larger
is drafted for a person to decide. The limit belongs to our code, so it goes in `deps`. `RefundDesk` is a `Desk` with
one more field, which is what the `(Desk)` after its name means, so every lookup written for a `Desk` accepts it.

In [14]:
@dataclass
class RefundDesk(Desk):
    approve_limit: float = 10.00


REFUND_DESK = RefundDesk(db=DESK.db)
print("deps fields:", list(RefundDesk.__dataclass_fields__))
print("approve limit:", REFUND_DESK.approve_limit)

deps fields: ['db', 'approve_limit']
approve limit: 10.0


### One more lookup: the invoice

A refund is about one invoice, so the agent needs to read one. We call it by hand on invoice 404.

In [15]:
def get_invoice(ctx: RunContext[Desk], invoice_id: int) -> dict | str:
    """Look up one invoice: the customer it belongs to, its date and its total in US dollars."""
    rows = query(ctx.deps, "SELECT InvoiceId, CustomerId, DATE(InvoiceDate), Total FROM invoices WHERE InvoiceId = ?",
                 (invoice_id,))
    if not rows:
        return f"no match for invoice {invoice_id}"
    return {"invoice_id": rows[0][0], "customer_id": rows[0][1], "date": rows[0][2], "total": rows[0][3]}


print(get_invoice(SimpleNamespace(deps=REFUND_DESK), 404))

{'invoice_id': 404, 'customer_id': 6, 'date': '2013-11-13', 'total': 25.86}


Invoice 404 belongs to customer 6, who is Helena, and its total is \$25.86. That is above the \$10 limit, so the
right decision for this request is a draft.

### The decision as a type, and the validator

`RefundDecision` has three fields. `action` is a `Literal`, so it can only be `approve` or `draft`.

The validator applies two rules, and both read the invoice **from the database**, with the same `get_invoice` the
model uses:

1. the refund cannot be larger than the invoice's total;
2. a refund above the limit cannot be approved, only drafted.

As in the playground, it prints what it was given, and each `ModelRetry` says what would pass.

Look at the instructions too. They explain the two actions. They do not mention the limit.

In [16]:
from typing import Literal


class RefundDecision(BaseModel):
    invoice_id: int
    amount: float
    action: Literal["approve", "draft"]


refunds = Agent(
    MODEL_NAME,
    output_type=RefundDecision,
    deps_type=RefundDesk,
    tools=[find_customer, get_invoice],
    instructions=INSTRUCTIONS + " For a refund request, answer with a decision: 'approve' pays the refund now, "
                                "'draft' writes it up for a person to decide.",
    model_settings=SETTINGS,
)


@refunds.output_validator
def check_refund(ctx: RunContext[RefundDesk], decision: RefundDecision) -> RefundDecision:
    print("validator saw:", decision.action, "|", decision.amount, "| invoice", decision.invoice_id)

    invoice = get_invoice(ctx, decision.invoice_id)
    if isinstance(invoice, str):
        raise ModelRetry(invoice)

    if decision.amount > invoice["total"]:
        raise ModelRetry(f"Invoice {decision.invoice_id} is for ${invoice['total']:.2f}. A refund cannot be larger.")

    if decision.amount > ctx.deps.approve_limit and decision.action == "approve":
        raise ModelRetry(f"Refunds over ${ctx.deps.approve_limit:.2f} need a person. Use action 'draft'.")

    return decision

`isinstance(invoice, str)` is true when the lookup answered `no match for invoice ...`: the decision names an
invoice that does not exist, and that sentence goes back to the model.

The rep's message ends with *Approve it.* Before you run the cell: what will the first `validator saw` line say?

In [17]:
REFUND = "Helena Holý asks for a refund of invoice 404. Approve it."

result = await refunds.run(REFUND, deps=REFUND_DESK)

print()
print("decision:      ", result.output)
print("model requests:", result.usage.requests)
print()
show_steps(result)

validator saw: approve | 25.86 | invoice 404


validator saw: draft | 25.86 | invoice 404

decision:       invoice_id=404 amount=25.86 action='draft'
model requests: 3

model asked for  : find_customer {"name":"Helena Holý"}
model asked for  : get_invoice {"invoice_id":404}
our code returned: [{'customer_id': 6, 'name': 'Helena Holý', 'country': 'Czech Republic'}]
our code returned: {'invoice_id': 404, 'customer_id': 6, 'date': '2013-11-13', 'total': 25.86}
model asked for  : final_result {"invoice_id":404,"amount":25.86,"action":"approve"}
our code returned: Refunds over $10.00 need a person. Use action 'draft'.
model asked for  : final_result {"invoice_id":404,"amount":25.86,"action":"draft"}
our code returned: Final result processed.


Two `validator saw` lines. The first decision was `approve` for \$25.86: the full invoice, approved, as the rep asked.
The validator read invoice 404 from the database, found the amount above the limit and raised `ModelRetry`. The second
decision is `draft` for the same amount, and that is the one we received.

The steps tell the same story from the model's side. It looked up the customer and the invoice in one reply, called
`final_result` with `approve`, was sent *Refunds over \$10.00 need a person. Use action 'draft'.* as that call's
result, and called `final_result` again with `draft`. That makes 3 model requests.

**What made the difference?** The model did what the rep asked. The rep wrote *Approve it*, the instructions said what
`approve` means, and nothing in the request or the instructions said that \$25.86 is too much. The limit is a field of
`deps`, and the validator checks it on every answer, whoever is asking and whatever they type.

**Important:** the validator did not take the model's word for the amount that matters. It read invoice 404 from the
database and compared the decision with that record. In the playground the validator had to trust the rating the model
wrote. Here the fact has a record, so the record is what we check against.

**In production:** put the limit in the instructions as well, so that the first decision is usually right, and keep
the validator. And keep the last step separate: an accepted `approve` is still only an object until code we trust acts
on it.

**Checkpoint C.** `validator saw` prints twice for the request that ends *Approve it*: `approve | 25.86` first, then
`draft | 25.86`. The decision we receive is the draft. In the steps, the reply to the first `final_result` is the
validator's sentence about refunds over \$10.00.

---

## Part D: Test the refund rule with no API call

Part C showed the rule working on one request, with a real model. A rule about money deserves tests that do not
depend on what a model writes today, and that cost nothing to run.

As in the playground, `TestModel` stands in for the model and answers with the fields we give it. Two details:

- `call_tools=[]` tells it not to call the lookups first. We are testing the validator, and the validator does its
  own reading of the database.
- `models.ALLOW_MODEL_REQUESTS = False` makes any real request raise an error while the tests run.

Each case is a decision and what must happen to it. A blocked decision ends the run with `UnexpectedModelBehavior`,
because a `TestModel` repeats the same answer until the retries run out.

In [18]:
from pydantic_ai import models
from pydantic_ai.exceptions import UnexpectedModelBehavior
from pydantic_ai.models.test import TestModel

CASES = [
    ({"invoice_id": 404, "amount": 25.86, "action": "approve"}, "blocked"),
    ({"invoice_id": 404, "amount": 25.86, "action": "draft"}, "returned"),
    ({"invoice_id": 46, "amount": 8.91, "action": "approve"}, "returned"),
    ({"invoice_id": 46, "amount": 99.00, "action": "approve"}, "blocked"),
]

models.ALLOW_MODEL_REQUESTS = False

for decision, expected in CASES:
    with refunds.override(model=TestModel(call_tools=[], custom_output_args=decision)):
        try:
            await refunds.run("a test", deps=REFUND_DESK)
            outcome = "returned"
        except UnexpectedModelBehavior:
            outcome = "blocked"
    print("✅" if outcome == expected else "❌", decision["action"], decision["amount"], "on invoice", decision["invoice_id"], "->", outcome, "\n")

models.ALLOW_MODEL_REQUESTS = True

validator saw: approve | 25.86 | invoice 404
validator saw: approve | 25.86 | invoice 404
✅ approve 25.86 on invoice 404 -> blocked 

validator saw: draft | 25.86 | invoice 404
✅ draft 25.86 on invoice 404 -> returned 

validator saw: approve | 8.91 | invoice 46
✅ approve 8.91 on invoice 46 -> returned 

validator saw: approve | 99.0 | invoice 46
validator saw: approve | 99.0 | invoice 46
✅ approve 99.0 on invoice 46 -> blocked 



Four ✅ lines. A blocked case shows two `validator saw` lines (the answer and its one retry), a returned case one. What
each case establishes:

| Decision | Outcome | What it establishes |
|---|---|---|
| approve \$25.86 on invoice 404 | blocked | an approval above the limit never reaches `result.output` |
| draft \$25.86 on invoice 404 | returned | the same refund as a draft is accepted |
| approve \$8.91 on invoice 46 | returned | the rule does not block what the desk allows |
| approve \$99.00 on invoice 46 | blocked | an amount above the invoice's \$8.91 is refused, whatever the model wrote |

The two `returned` rows matter as much as the two `blocked` ones. A validator that refused everything would pass both
blocked cases.

These tests say what our rule does with a given decision. They say nothing about which decision a real model will
propose; Part C's run is the evidence for that, and it is one run.

**Checkpoint D.** Four ✅ lines, in the order blocked, returned, returned, blocked, with no model request.

---

## What you can do now

- Pass the database to tools through `deps`, with a connection per lookup, and say why one shared connection is
  unsafe here (Part A).
- Get a typed record from three lookups, and show from the messages that the lookups ran (Part A).
- Give an agent a second output type so that it can answer "not found" (Part B).
- Check a decision against a database record in an output validator, and send it back with the reason (Part C).
- Test the validator with `TestModel`, on cases it must block and cases it must pass (Part D).

## Troubleshooting

| What you see | Why | Fix |
|---|---|---|
| `AttributeError: 'NoneType' object has no attribute 'db'` | the run was started without `deps=` | pass `deps=DESK` (or `deps=REFUND_DESK` for the refund agent) |
| `sqlite3.OperationalError: unable to open database file` | the address in `Desk.db` does not point at `data/chinook.db` | run the setup cell from inside the repository, so that `ROOT` is found |
| `RuntimeError: Model requests are not allowed, since ALLOW_MODEL_REQUESTS is False` | Part D's switch is still off, usually after an interrupted cell | run `models.ALLOW_MODEL_REQUESTS = True` |
| `UnexpectedModelBehavior: Exceeded maximum output retries (1)` on a real run | the validator refused the answer and its retry | read the `validator saw` lines and make the `ModelRetry` message say what would pass |

## Challenge

1. **Puja at the desk.** Ask the desk agent from Part B for Puja Srivastava's invoices, total and support rep. Then
   ask again with her surname misspelt as *Shrivastava*. Print the type of each answer.
2. **A refund for the wrong customer.** A decision should say whose refund it is. Add a `customer_id` field to a new
   decision type, and a third rule to a new validator: the invoice must belong to that customer. Ask for
   *Puja Srivastava asks for a refund of invoice 229. Approve it.*, then test the new rule with `TestModel`: a draft
   of \$25.86 on invoice 404 for customer 59 must be blocked, and the same draft for customer 6 returned.

Expected: task 1 gives a `DeskAnswer` (59, 6 invoices, \$36.64, Jane Peacock) and then a `NotFound`. In task 2 the
real run ends in a draft of \$13.86 for customer 59 after one correction, and the test prints one ✅ for the
blocked case and one for the returned case.

The solution follows.

---

## Challenge solution

### 1. Puja at the desk

The same agent, two spellings. Our code reads the type of the answer to tell the cases apart.

In [19]:
for name in ["Puja Srivastava", "Puja Shrivastava"]:
    question = f"How many invoices does {name} have, what do they add up to, and who is her support rep?"
    result = await desk.run(question, deps=DESK)
    print(type(result.output).__name__, "|", result.output)

DeskAnswer | customer_id=59 invoice_count=6 total=36.64 support_rep='Jane Peacock'


NotFound | searched_for='Puja Shrivastava'


The right spelling gives a `DeskAnswer` with Puja's four values from Part A's table: customer 59, 6 invoices,
\$36.64, Jane Peacock. The misspelt surname gives a `NotFound` that carries the name as it was typed.

### 2. A refund for the wrong customer

The new decision type adds `customer_id`. The new validator keeps the two rules of Part C and adds one that reads
the invoice's owner from the database.

In [20]:
class CustomerRefund(BaseModel):
    customer_id: int
    invoice_id: int
    amount: float
    action: Literal["approve", "draft"]


customer_refunds = Agent(
    MODEL_NAME,
    output_type=CustomerRefund,
    deps_type=RefundDesk,
    tools=[find_customer, get_invoice],
    instructions=INSTRUCTIONS + " For a refund request, answer with a decision: 'approve' pays the refund now, "
                                "'draft' writes it up for a person to decide.",
    model_settings=SETTINGS,
)


@customer_refunds.output_validator
def check_customer_refund(ctx: RunContext[RefundDesk], decision: CustomerRefund) -> CustomerRefund:
    print("validator saw:", decision.action, "|", decision.amount, "| invoice", decision.invoice_id,
          "| customer", decision.customer_id)

    invoice = get_invoice(ctx, decision.invoice_id)
    if isinstance(invoice, str):
        raise ModelRetry(invoice)

    if invoice["customer_id"] != decision.customer_id:
        raise ModelRetry(f"Invoice {decision.invoice_id} does not belong to customer {decision.customer_id}.")

    if decision.amount > invoice["total"]:
        raise ModelRetry(f"Invoice {decision.invoice_id} is for ${invoice['total']:.2f}. A refund cannot be larger.")

    if decision.amount > ctx.deps.approve_limit and decision.action == "approve":
        raise ModelRetry(f"Refunds over ${ctx.deps.approve_limit:.2f} need a person. Use action 'draft'.")

    return decision


result = await customer_refunds.run("Puja Srivastava asks for a refund of invoice 229. Approve it.", deps=REFUND_DESK)

print()
print("decision:      ", result.output)
print("model requests:", result.usage.requests)

validator saw: approve | 13.86 | invoice 229 | customer 59


validator saw: draft | 13.86 | invoice 229 | customer 59

decision:       customer_id=59 invoice_id=229 amount=13.86 action='draft'
model requests: 3


The same pattern as Part C, on another customer: `approve` for \$13.86 first, because the rep asked for it, then
`draft` after the validator's sentence about the limit. Both decisions name customer 59, which is the owner of
invoice 229, so the new rule had nothing to refuse in this run. That is why it needs a test of its own.

Now the test of the new rule. Invoice 404 is Helena's (customer 6), so a draft that names customer 59 must be
blocked, and the same draft for customer 6 returned.

In [21]:
CASES = [
    ({"customer_id": 59, "invoice_id": 404, "amount": 25.86, "action": "draft"}, "blocked"),
    ({"customer_id": 6, "invoice_id": 404, "amount": 25.86, "action": "draft"}, "returned"),
]

models.ALLOW_MODEL_REQUESTS = False

for decision, expected in CASES:
    with customer_refunds.override(model=TestModel(call_tools=[], custom_output_args=decision)):
        try:
            await customer_refunds.run("a test", deps=REFUND_DESK)
            outcome = "returned"
        except UnexpectedModelBehavior:
            outcome = "blocked"
    print("✅" if outcome == expected else "❌", decision["action"], decision["amount"], "on invoice", decision["invoice_id"], "->", outcome, "\n")

models.ALLOW_MODEL_REQUESTS = True

validator saw: draft | 25.86 | invoice 404 | customer 59
validator saw: draft | 25.86 | invoice 404 | customer 59
✅ draft 25.86 on invoice 404 -> blocked 

validator saw: draft | 25.86 | invoice 404 | customer 6
✅ draft 25.86 on invoice 404 -> returned 



The draft for customer 59 was blocked: the validator read invoice 404, found that it belongs to customer 6, and
refused the decision and its retry. The same draft for customer 6 was returned. Neither case made a model request.

## Going further

- **A scripted model.** `TestModel` repeats one answer, so it can test that a decision is blocked or returned.
  To test the correction itself (an approval first, a draft after the retry), the testing guide has
  `FunctionModel`: a stand-in whose answers we write as a Python function that receives the messages so far.
- **Approval before a tool runs.** This notebook checks the agent's final answer. Pydantic AI can also pause a run
  when the model asks for a tool that needs a person's approval, and continue once the answer is in. Its
  documentation calls these deferred tools.
- **Seeing runs outside a notebook.** Pydantic AI can send a trace of every run to Logfire, a hosted service from
  the same company. It needs an account, so this lab does not use it.

## Resources

- [Output](https://pydantic.dev/docs/ai/core-concepts/output/): `output_type` with one type or several, output
  validators
- [Dependencies](https://pydantic.dev/docs/ai/core-concepts/dependencies/): `deps_type`, `deps=` and `RunContext`
- [Function tools](https://pydantic.dev/docs/ai/tools-toolsets/tools/): tools as plain functions, `tools=[...]` and
  the decorators
- [Retries](https://pydantic.dev/docs/ai/core-concepts/retries/): what `ModelRetry` does and how many tries a run
  gets
- [Testing](https://pydantic.dev/docs/ai/guides/testing/): `TestModel`, `FunctionModel`, `Agent.override` and
  `ALLOW_MODEL_REQUESTS`
- [Deferred tools](https://pydantic.dev/docs/ai/tools-toolsets/deferred-tools/): tool calls that wait for a person's
  approval
- [Python `sqlite3`](https://docs.python.org/3/library/sqlite3.html): connections, read-only addresses and threads

Back to [the lab's README](../README.md).